In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy

In [ ]:
#Import statements
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils import resample

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
#Running RoBERTa with CPU is very slow, so want to use GPU
print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
#Loading the dataset
FILE_PATH = "/content/Sentences_75Agree.txt"

with open(FILE_PATH, "r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

#Parse text and labels from "sentence@label" format
rows = []
for line in lines:
    line = line.strip()
    if not line:
        continue

    parts = line.rsplit("@", 1)
    if len(parts) != 2:
        continue

    text, label = parts
    text = text.strip()
    label = label.strip().lower()

    #Only keep valid labels
    if text and label in ["positive", "neutral", "negative"]:
        rows.append((text, label))

#Convert to dataframe
df = pd.DataFrame(rows, columns=["text", "label"])

print("Shape:", df.shape)
print(df["label"].value_counts())

In [ ]:
#Encoding the labels

#Convert labels (pos/neutral/neg) to numeric (0,1,2)
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

print("Classes:", list(label_encoder.classes_))

X = df["text"].tolist()
y = df["label_id"].tolist()

In [ ]:
#Splitting into pool and test sets

#Creates held-out test set that's not touched during training
X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Pool size:", len(X_pool))
print("Test size:", len(X_test))

In [ ]:
#Combine pool into dataframe for easier sampling (creating pool dataframe)
pool_df = pd.DataFrame({
    "text": X_pool,
    "label": y_pool
})

print("\nPool class counts:")
print(pool_df["label"].value_counts().sort_index())

In [ ]:
#Balancing initial seed set (instead of random sampling, want to force equal samples
#per class)
INITIAL_PER_CLASS = 60
RANDOM_STATE = 42

labeled_parts = []
remaining_parts = []

for cls in sorted(pool_df["label"].unique()):
    class_subset = pool_df[pool_df["label"] == cls]

    #Sample equal number from each class
    sampled = class_subset.sample(
        n=INITIAL_PER_CLASS,
        random_state=RANDOM_STATE
    )
    labeled_parts.append(sampled)

    #Remaining data becomes unlabeled pool
    remaining = class_subset.drop(sampled.index)
    remaining_parts.append(remaining)

#Shuffle
labeled_df = pd.concat(labeled_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
remaining_df = pd.concat(remaining_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nBalanced labeled set class counts:")
print(labeled_df["label"].value_counts().sort_index())

print("\nRemaining unlabeled pool class counts:")
print(remaining_df["label"].value_counts().sort_index())

In [ ]:
#Oversample training set

#Make training data balanced by duplicating the minority classes
max_class_size = labeled_df["label"].value_counts().max()

oversampled_parts = []
for cls in sorted(labeled_df["label"].unique()):
    cls_df = labeled_df[labeled_df["label"] == cls]

    cls_upsampled = resample(
        cls_df,
        replace=True,
        n_samples=max_class_size,
        random_state=RANDOM_STATE
    )
    oversampled_parts.append(cls_upsampled)

train_balanced_df = pd.concat(oversampled_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nOversampled training class counts:")
print(train_balanced_df["label"].value_counts().sort_index())

In [ ]:
#Final train and test sets
train_df = pd.DataFrame({
    "text": train_balanced_df["text"],
    "label": train_balanced_df["label"]
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

print("\nTrain size after oversampling:", len(train_df))
print("Test size:", len(test_df))

In [ ]:
#Tokenizer

#Converts text into tokens for RoBERTa
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

#Removes raw text column (not needed anymore)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

#Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
#Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

In [ ]:
#Model: loads pretrained RoBERTa and adds classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_)
)

In [ ]:
#Training arguments
training_args = TrainingArguments(
    output_dir="./roberta_balanced_results",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    report_to="none"
)

In [ ]:
#Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
#Final evaluation
eval_results = trainer.evaluate()
print("\nEval results:")
print(eval_results)

pred_output = trainer.predict(test_dataset)
preds = np.argmax(pred_output.predictions, axis=1)

print("\nMacro F1:", f1_score(y_test, preds, average="macro"))
print("Accuracy:", accuracy_score(y_test, preds))
print("\nClassification report:\n")
print(classification_report(
    y_test,
    preds,
    target_names=label_encoder.classes_,
    zero_division=0
))